# Práctica 1. Clasificación de reseñas de películas

Redes Neuronales para el Procesamiento del Lenguaje Natural

El objetivo es clasificar reseñas como positivas o negativas. Se mantiene la partición proporcionada de 8.000 reseñas de entrenamiento y validación y 2.000 de test. La selección de representaciones y parámetros utiliza exclusivamente las primeras 8.000.

Se sigue la secuencia de la práctica guiada del tema 1, adaptada de noticias a reseñas: inspección, limpieza, representaciones y clasificación. Los requisitos proceden del enunciado de la práctica.

## 1. Carga y análisis de los datos

El dataset docente debe estar en `materiales/Dataset_reviews.zip`, junto al cuaderno, o en `Dataset_reviews.zip` en el directorio de ejecución, por ejemplo tras subirlo a Colab. Se lee el ZIP directamente, sin modificar sus archivos. Se necesitan NumPy, pandas, SciPy, scikit-learn y Matplotlib, además de NLTK para la comparación con stemming y TensorFlow/Keras para las redes neuronales. En Colab, si faltase alguna dependencia, puede instalarse con `%pip install tensorflow-cpu scikit-learn nltk`. No se descargan corpus ni modelos externos.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import html
import json
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from nltk.stem import PorterStemmer

SEED = 42
LABELS = ["negative", "positive"]
DATASET = next((p for p in [Path("materiales/Dataset_reviews.zip"), Path("Dataset_reviews.zip")]
                if p.is_file()), None)
if DATASET is None:
    raise FileNotFoundError("Coloque Dataset_reviews.zip en materiales/ o junto al cuaderno.")
with ZipFile(DATASET) as z:
    train = pd.read_csv(z.open("Dataset_reviews/train_reviews.csv"))
    test = pd.read_csv(z.open("Dataset_reviews/test_reviews.csv"))
dataset_sha256 = hashlib.sha256(DATASET.read_bytes()).hexdigest()
assert dataset_sha256 == "1d38a658b1390ed567c2ecfa8104b4c58f315f0f5761a96cd00fe936b6a47b94"
assert len(train) == 8000 and len(test) == 2000
for frame in [train, test]:
    assert list(frame.columns) == ["id", "review", "sentiment"]
    assert frame.notna().all().all()
    assert frame.id.is_unique
    assert set(frame.sentiment) == set(LABELS)
assert set(train.id).isdisjoint(test.id)

display(train.head(3))
balance = pd.DataFrame({"entrenamiento y validación": train.sentiment.value_counts(),
                        "test": test.sentiment.value_counts()}).loc[LABELS]
display(balance)

In [ ]:
def auditar(frame):
    lengths = frame.review.str.split().str.len()
    return {
        "reseñas": len(frame),
        "celdas nulas": int(frame.isna().sum().sum()),
        "textos vacíos": int(frame.review.str.strip().eq("").sum()),
        "IDs distintos": frame.id.nunique(),
        "duplicados de texto adicionales": int(frame.review.duplicated().sum()),
        "grupos de texto con etiquetas contradictorias": int(
            (frame.groupby("review").sentiment.nunique() > 1).sum()),
        "palabras media": lengths.mean(),
        "palabras mediana": lengths.median(),
        "palabras mínimo": lengths.min(),
        "palabras máximo": lengths.max(),
        "textos con HTML": int(frame.review.str.contains(r"<[^>]+>", regex=True).sum()),
    }

auditoria = pd.DataFrame({"entrenamiento y validación": auditar(train), "test": auditar(test)})
display(auditoria.round(2))
raw_shared = set(train.review) & set(test.review)
print("Textos exactos compartidos entre los archivos:", len(raw_shared))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
balance["entrenamiento y validación"].plot.bar(ax=axes[0], rot=0, color=["#ad4b45", "#377d6a"])
axes[0].set(title="Clases en las 8.000 reseñas", ylabel="Reseñas", xlabel="Sentimiento")
axes[1].hist(train.review.str.split().str.len(), bins=35, color="#386a99")
axes[1].set(title="Longitud de las 8.000 reseñas", xlabel="Palabras separadas por espacios", ylabel="Reseñas")
plt.tight_layout()
plt.show()

Las clases de entrenamiento están casi equilibradas, por lo que no se aplica sobremuestreo ni ponderación de clases. Las longitudes varían mucho y aparecen marcas HTML. La mediana complementa a la media para describir la asimetría.

No hay IDs compartidos entre archivos, pero sí ocho textos repetidos. Se conservan las 8.000/2.000 filas exigidas. Esa repetición puede favorecer la evaluación final, incluso con IDs diferentes, y deberá tenerse en cuenta al interpretar el test. Las comprobaciones de test de este apartado se limitan a estructura, calidad y distribución; no se evalúan predicciones ni se eligen parámetros con él.

## 2. Limpieza y separación de validación

Se decodifican entidades HTML, se sustituyen las etiquetas por espacios, se pasa a minúsculas, se expanden las negaciones contraídas y se conservan las letras. La normalización no aprende estadísticas del corpus. Se mantienen las palabras funcionales y las negaciones porque expresiones como `not good` pueden cambiar la polaridad. Esta elección se contrasta después con una variante que elimina stopwords y aplica stemming.

El vocabulario y el IDF se ajustarán dentro de cada partición de entrenamiento. En la guiada, las representaciones se calculan antes de la validación cruzada del clasificador; aquí se incluyen en un `Pipeline` para que los pliegues de validación tampoco intervengan en esas estimaciones.

In [ ]:
def limpiar(text):
    text = html.unescape(text)
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.lower().replace("’", "'")
    for contraction, expansion in {"can't": "can not", "cannot": "can not",
                                    "won't": "will not", "shan't": "shall not",
                                    "ain't": "is not"}.items():
        text = re.sub(r"\b" + re.escape(contraction) + r"\b", expansion, text)
    text = re.sub(r"\b([a-z]+)n't\b", r"\1 not", text)
    return " ".join(re.sub(r"[^a-z]+", " ", text).split())

train["clean"] = train.review.map(limpiar)
test["clean"] = test.review.map(limpiar)
assert train.clean.str.len().gt(0).all() and test.clean.str.len().gt(0).all()
assert limpiar("I can't say it's good.<br />It isn't!") == "i can not say it s good it is not"

clean_audit = pd.DataFrame({
    "entrenamiento y validación": [train.clean.nunique(), int(train.clean.duplicated().sum()),
        int((train.groupby("clean").sentiment.nunique() > 1).sum())],
    "test": [test.clean.nunique(), int(test.clean.duplicated().sum()),
        int((test.groupby("clean").sentiment.nunique() > 1).sum())]},
    index=["textos normalizados distintos", "duplicados normalizados adicionales",
           "grupos normalizados con etiquetas contradictorias"])
display(clean_audit)
clean_shared = set(train.clean) & set(test.clean)
print("Textos normalizados compartidos entre archivos:", len(clean_shared))
display(train.loc[:1, ["review", "clean"]])

# Primer pliegue de cinco: la regla y la semilla se fijan antes de experimentar.
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
fit_idx, val_idx = next(outer.split(train.clean, train.sentiment, groups=train.clean))
fit = train.iloc[fit_idx].copy()
valid = train.iloc[val_idx].copy()
assert len(fit) == 6400 and len(valid) == 1600
assert set(fit.id).isdisjoint(valid.id)
assert set(fit.clean).isdisjoint(valid.clean)
assert set(fit_idx) | set(val_idx) == set(range(8000))
assert set(fit.id) | set(valid.id) == set(train.id)
display(pd.DataFrame({"ajuste": fit.sentiment.value_counts(),
                      "validación": valid.sentiment.value_counts()}).loc[LABELS])

La limpieza deja 10 duplicados adicionales en entrenamiento y uno en test, sin grupos de etiquetas contradictorias. No se eliminan filas. Las reseñas con el mismo texto normalizado se asignan al mismo grupo para impedir que crucen entre ajuste y validación. Con la semilla 42, el primer pliegue deja 6.400 reseñas para ajuste y 1.600 para validación. Esta separación también se utilizará con las redes neuronales.

Los parámetros clásicos se exploran con tres pliegues internos, estratificados por clase y separados por grupo, dentro de las 6.400 reseñas. Las 1.600 de validación permiten comparar las configuraciones elegidas y, posteriormente, las redes. Al usarlas para comparar modelos, su puntuación forma parte del desarrollo; no constituye una estimación final independiente.

## 3. Representaciones vectoriales

Se comparan presencia binaria por término, frecuencias absolutas y TF-IDF con unigramas. La presencia binaria es una bolsa de palabras multihot por documento, no un único vector one-hot. Se añade TF-IDF con unigramas y bigramas para representar combinaciones como `not good`.

En todas las variantes se exige presencia en al menos dos documentos del ajuste y se limita el vocabulario a 30.000 términos para contener el coste y reducir términos muy raros. Los unigramas comparten tokenización y criterios de vocabulario; TF-IDF añade IDF y normalización L2. Por tanto, su diferencia frente a frecuencias no permite atribuir el cambio de resultado exclusivamente al IDF. Los bigramas cambian también el espacio de características. No se elimina ningún término por frecuencia máxima.

In [ ]:
COMMON = dict(min_df=2, max_df=1.0, max_features=30000,
              lowercase=False, token_pattern=r"(?u)\b[a-z][a-z]+\b")
representaciones = {
    "Binaria 1g": CountVectorizer(**COMMON, binary=True, ngram_range=(1, 1)),
    "Frecuencias 1g": CountVectorizer(**COMMON, binary=False, ngram_range=(1, 1)),
    "TF-IDF 1g": TfidfVectorizer(**COMMON, ngram_range=(1, 1), norm="l2", smooth_idf=True),
    "TF-IDF 1-2g": TfidfVectorizer(**COMMON, ngram_range=(1, 2), norm="l2", smooth_idf=True),
}

representacion_info = []
for name, vectorizer in representaciones.items():
    vec = clone(vectorizer)
    matrix = vec.fit_transform(fit.clean)
    representacion_info.append({"representación": name, "documentos de ajuste": matrix.shape[0],
        "términos": matrix.shape[1], "entradas no nulas": matrix.nnz,
        "densidad": matrix.nnz / (matrix.shape[0] * matrix.shape[1])})
display(pd.DataFrame(representacion_info).set_index("representación").round(5))

## 4. Algoritmos clásicos

Se prueban Naïve Bayes multinomial, regresión logística y SVM lineal. Naïve Bayes ofrece una referencia probabilística; regresión logística y SVM aprenden fronteras lineales apropiadas para matrices dispersas. Se usa `LinearSVC` para evitar el coste de una SVM con kernel en esta comparación.

Se explora `alpha` en {0,1; 1; 10} para el suavizado de Naïve Bayes y `C` en {0,1; 1; 10} para la regularización de regresión logística y SVM. Se mantiene penalización L2 en los modelos lineales. Las 36 configuraciones resultan de cuatro representaciones, tres algoritmos y tres valores de parámetro. F1 macro es el criterio principal porque da el mismo peso a ambas clases. Se informa también de accuracy. La búsqueda elige el mayor F1 macro medio de tres pliegues; los empates conservan el primer valor del orden declarado.

Se registran todos los resultados, los parámetros elegidos y los tiempos. La desviación entre tres pliegues describe la variación de esa validación cruzada y no es un intervalo de confianza.

In [ ]:
algoritmos = {
    "Naïve Bayes": (MultinomialNB(), {"clf__alpha": [0.1, 1.0, 10.0]}),
    "Regresión logística": (LogisticRegression(solver="liblinear", max_iter=3000,
        tol=1e-4, random_state=SEED), {"clf__C": [0.1, 1.0, 10.0]}),
    "SVM lineal": (LinearSVC(dual="auto", max_iter=10000, tol=1e-4, random_state=SEED),
        {"clf__C": [0.1, 1.0, 10.0]}),
}
inner = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
inner_splits = list(inner.split(fit.clean, fit.sentiment, groups=fit.clean))
for tr, va in inner_splits:
    assert set(fit.clean.iloc[tr]).isdisjoint(fit.clean.iloc[va])

cv_rows, selected_rows, models = [], [], {}
warnings.filterwarnings("error", category=ConvergenceWarning)
for rep_name, vectorizer in representaciones.items():
    for alg_name, (estimator, grid) in algoritmos.items():
        start = time.perf_counter()
        pipe = Pipeline([("vec", clone(vectorizer)), ("clf", clone(estimator))])
        search = GridSearchCV(pipe, grid, scoring={"f1_macro": "f1_macro", "accuracy": "accuracy"},
            refit="f1_macro", cv=inner_splits, n_jobs=2, error_score="raise", return_train_score=False)
        search.fit(fit.clean, fit.sentiment)
        elapsed = time.perf_counter() - start
        res = search.cv_results_
        for i, params in enumerate(res["params"]):
            cv_rows.append({"representación": rep_name, "algoritmo": alg_name,
                "parámetros": json.dumps(params, sort_keys=True),
                "F1 macro CV": float(res["mean_test_f1_macro"][i]),
                "desv. F1 CV": float(res["std_test_f1_macro"][i]),
                "accuracy CV": float(res["mean_test_accuracy"][i]),
                "ajuste medio s": float(res["mean_fit_time"][i]),
                "F1 por pliegue": [float(res[f"split{k}_test_f1_macro"][i]) for k in range(3)]})
        model = search.best_estimator_
        # Verifica que el refit final utiliza exactamente el vocabulario del ajuste.
        assert model.named_steps["vec"].vocabulary_ == clone(vectorizer).fit(fit.clean).vocabulary_
        prediction = model.predict(valid.clean)
        assert len(prediction) == 1600
        row = {"representación": rep_name, "algoritmo": alg_name,
            "parámetros": json.dumps(search.best_params_, sort_keys=True),
            "F1 macro CV": float(search.best_score_),
            "F1 macro validación": float(f1_score(valid.sentiment, prediction, average="macro")),
            "accuracy validación": float(accuracy_score(valid.sentiment, prediction)),
            "búsqueda y refit s": elapsed}
        selected_rows.append(row)
        models[(rep_name, alg_name)] = model
        print(f"{rep_name} / {alg_name}: F1 validación={row['F1 macro validación']:.4f}", flush=True)

cv_results = pd.DataFrame(cv_rows)
comparison = pd.DataFrame(selected_rows).sort_values(
    ["F1 macro validación", "accuracy validación"], ascending=False, kind="stable").reset_index(drop=True)
assert len(cv_results) == 36 and len(comparison) == 12
assert cv_results["F1 macro CV"].between(0, 1).all()
assert comparison["F1 macro validación"].between(0, 1).all()
display(comparison.round(4))
display(cv_results.drop(columns="F1 por pliegue").round(4))

In [ ]:
best = comparison.iloc[0]
best_key = (best["representación"], best["algoritmo"])
best_model = models[best_key]
pred_valid = best_model.predict(valid.clean)
display(pd.DataFrame(confusion_matrix(valid.sentiment, pred_valid, labels=LABELS),
                     index=["real negativa", "real positiva"],
                     columns=["predicha negativa", "predicha positiva"]))
report = classification_report(valid.sentiment, pred_valid, labels=LABELS, output_dict=True)
display(pd.DataFrame(report).T.round(4))
majority = fit.sentiment.mode().iloc[0]
majority_pred = np.repeat(majority, len(valid))
display(pd.DataFrame([{"referencia": "clase mayoritaria del ajuste",
    "accuracy validación": accuracy_score(valid.sentiment, majority_pred),
    "F1 macro validación": f1_score(valid.sentiment, majority_pred, average="macro")}]).round(4))

display(Markdown(
    f"La mejor combinación clásica en esta validación es **{best_key[1]} con {best_key[0]}**, "
    f"con {best['parámetros']}. Obtiene F1 macro {best['F1 macro validación']:.4f} y "
    f"accuracy {best['accuracy validación']:.4f}. La selección sigue abierta hasta comparar "
    "las redes neuronales. Las diferencias pequeñas entre variantes deben interpretarse con "
    "cautela porque proceden de las mismas 1.600 reseñas y de una búsqueda limitada."))

Con regresión logística y `C=10`, añadir bigramas a TF-IDF aumenta la accuracy de validación de 0,8625 a 0,8750, una diferencia de 20 aciertos entre 1.600 reseñas. La SVM y Naïve Bayes también obtienen su mejor resultado de validación con TF-IDF de unigramas y bigramas. Esto respalda conservar esa representación como referencia clásica para la comparación con las redes, sin demostrar que sea superior en otros datos.

El mejor `C` de regresión logística llega al extremo superior de la rejilla. La búsqueda es acotada y no establece el óptimo del algoritmo. Naïve Bayes elige `alpha=10` en frecuencias y `alpha=0.1` en TF-IDF con bigramas; el suavizado necesita ajustarse a la escala y representación de los datos. El modelo clásico provisional comete 101 errores de negativa a positiva y 99 en sentido contrario, sin una diferencia marcada entre ambas direcciones.

### Comprobación de la limpieza

Se contrasta la limpieza principal con una variante inspirada en la guiada: eliminación de stopwords y stemming Porter. Se conservan `no`, `not` y `nor`. Se fija regresión logística con TF-IDF de unigramas y bigramas y `C=1`; no se vuelve a buscar su parámetro. La diferencia mide el efecto conjunto de eliminar stopwords y aplicar stemming, sin atribuirlo por separado a cada transformación. La lematización de WordNet de la guiada no se reproduce en esta variante.

In [ ]:
stemmer = PorterStemmer()
stops = set(ENGLISH_STOP_WORDS) - {"no", "not", "nor"}

def limpiar_con_stemming(clean_text):
    return " ".join(stemmer.stem(word) for word in clean_text.split() if word not in stops)

cleaning_rows = []
for name, train_text, valid_text in [
    ("Principal, conserva palabras funcionales", fit.clean, valid.clean),
    ("Sin stopwords salvo negaciones y con stemming", fit.clean.map(limpiar_con_stemming),
        valid.clean.map(limpiar_con_stemming))]:
    estimator = Pipeline([("vec", clone(representaciones["TF-IDF 1-2g"])),
        ("clf", LogisticRegression(C=1.0, solver="liblinear", max_iter=3000, random_state=SEED))])
    start = time.perf_counter()
    estimator.fit(train_text, fit.sentiment)
    predicted = estimator.predict(valid_text)
    cleaning_rows.append({"limpieza": name,
        "F1 macro validación": float(f1_score(valid.sentiment, predicted, average="macro")),
        "accuracy validación": float(accuracy_score(valid.sentiment, predicted)),
        "términos": len(estimator.named_steps["vec"].vocabulary_),
        "ajuste y predicción s": time.perf_counter() - start})
cleaning_results = pd.DataFrame(cleaning_rows)
display(cleaning_results.round(4))
delta = cleaning_results.iloc[1]["F1 macro validación"] - cleaning_results.iloc[0]["F1 macro validación"]
display(Markdown(f"El cambio conjunto de limpieza modifica F1 macro en {delta:+.4f} en esta validación. "
    "Esta comparación no demuestra una ventaja general de ninguna limpieza. La normalización principal "
    "conserva las formas léxicas y será el punto de partida común de los modelos neuronales."))

### Errores de validación

Se muestran las dos primeras equivocaciones de cada dirección de error, por orden del CSV. La regla evita escoger únicamente ejemplos que encajen con una explicación. Los fragmentos sirven para localizar las reseñas; la comparación con las redes neuronales se desarrolla en el apartado 6.

In [ ]:
errors = valid[["id", "review", "sentiment"]].copy()
errors["predicción"] = pred_valid
errors = errors[errors.sentiment != errors["predicción"]]
examples = errors.groupby("sentiment", sort=True).head(2).copy()
examples["fragmento"] = examples.review.str.slice(0, 650)
with pd.option_context("display.max_colwidth", 700):
    display(examples[["id", "sentiment", "predicción", "fragmento"]])
print("Errores en validación:", len(errors), "de", len(valid))
print("El test conserva sus", len(test), "reseñas y no se ha utilizado para predecir ni puntuar modelos.")

La reseña 19118 utiliza `great` entre comillas junto a una valoración claramente negativa, lo que ilustra la dificultad de interpretar ironía con bolsas de palabras. La 2895 describe experiencias del autor y circunstancias del rodaje, además de su valoración; la 6985 combina elogios con reparos sobre la duración y una subtrama. Son fenómenos presentes en errores reales, aunque su lectura por sí sola no permite atribuir la predicción a una palabra concreta. La 963 contiene elogios explícitos y también se clasifica mal, por lo que los errores no se limitan a textos ambiguos.

## 5. Redes neuronales básicas

Se mantiene la separación de 6.400 reseñas de ajuste y 1.600 de validación. Las redes reciben la misma limpieza principal que los modelos clásicos. Se comparan capas densas sobre el TF-IDF de unigramas y bigramas y embeddings aprendidos desde cero sobre tokens. No se usan vectores externos: así se estudia qué aprende cada representación con los mismos datos de ajuste.

La guiada emplea capas densas de 128 y 64 unidades y embeddings con `Flatten`. Aquí se conservan esas familias y se adapta la salida a una neurona sigmoide con entropía cruzada binaria y etiquetas 0/1. La sigmoide produce una probabilidad de clase positiva; el umbral se fija en 0,5. Las capas ocultas usan ReLU o tanh, en lugar del softmax oculto de la guiada, para comparar transformaciones no lineales sin imponer que sus activaciones sumen uno.

La búsqueda es acotada y se define antes de entrenar. La referencia densa tiene 64 unidades ReLU, Adam con tasa 0,001 y batch 64. Cada variante cambia un factor respecto a esa referencia: profundidad/anchura, activación, tasa, optimizador o batch. En embeddings se comparan 32 y 64 dimensiones y aplanado frente a promedio enmascarado, con una capa oculta de 32 unidades. La variante de batch cambia solo ese factor respecto al promedio de 32 dimensiones. Adam adapta los pasos por parámetro; SGD con momento sirve como contraste de optimización, con tasa 0,1 propia de ese optimizador. Ese contraste modifica optimizador y tasa conjuntamente.

Se limita el entrenamiento a 20 épocas, con parada si la pérdida de validación no mejora al menos 0,001 durante tres épocas. Se restauran los pesos de la mejor época aceptada por ese criterio de mejora, aunque su F1 no sea el mayor de todas las épocas. L2 de 0,0001 en las capas ocultas y dropout de 0,3 se mantienen fijos para limitar sobreajuste. F1 macro de validación ordena las variantes; accuracy resuelve empates y el orden declarado resuelve los restantes. No se busca el umbral en validación.

La misma validación controla la parada y permite comparar variantes, por lo que sus resultados pueden ser optimistas tras la búsqueda. Esta comparación orienta el desarrollo; el test permanece reservado para una única evaluación posterior a la selección.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from collections import Counter
import gc

tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.experimental.enable_op_determinism()
keras.utils.set_random_seed(SEED)
y_fit = (fit.sentiment == "positive").astype("float32").to_numpy()
y_valid = (valid.sentiment == "positive").astype("float32").to_numpy()
# El TF-IDF ya aprendido por la referencia clásica solo conoce las 6.400 de ajuste.
nn_vectorizer = models[("TF-IDF 1-2g", "Regresión logística")].named_steps["vec"]
X_dense_fit = nn_vectorizer.transform(fit.clean).astype("float32")
X_dense_valid = nn_vectorizer.transform(valid.clean).astype("float32")

MAX_WORDS, MAX_LEN = 20000, 512
# Empates léxicos para un vocabulario determinista; 0=padding y 1=desconocida.
def tokens(text):
    return re.findall(r"\b[a-z][a-z]+\b", text)
fit_tokens = [tokens(s) for s in fit.clean]
valid_tokens = [tokens(s) for s in valid.clean]
counts = Counter(word for row in fit_tokens for word in row)
ordered = sorted((w for w in counts if counts[w] >= 2), key=lambda w: (-counts[w], w))
word_index = {w: i+2 for i, w in enumerate(ordered[:MAX_WORDS-2])}
VOCAB_SIZE = len(word_index) + 2
assert set(word_index) <= set(counts)

def encode(rows):
    matrix = np.zeros((len(rows), MAX_LEN), dtype="int32")
    for i, row in enumerate(rows):
        ids = [word_index.get(w, 1) for w in row[:MAX_LEN]]
        matrix[i, :len(ids)] = ids
    return matrix

X_emb_fit, X_emb_valid = encode(fit_tokens), encode(valid_tokens)
sequence_audit = []
for name, rows, matrix in [("ajuste", fit_tokens, X_emb_fit), ("validación", valid_tokens, X_emb_valid)]:
    retained = int(np.count_nonzero(matrix))
    sequence_audit.append({"partición": name, "reseñas": len(rows),
        "vocabulario aprendido en ajuste": VOCAB_SIZE,
        "longitud máxima": MAX_LEN, "reseñas truncadas": sum(len(r) > MAX_LEN for r in rows),
        "tokens descartados": sum(max(0, len(r)-MAX_LEN) for r in rows),
        "tokens retenidos": retained, "tokens desconocidos": int((matrix == 1).sum()),
        "fracción desconocida": int((matrix == 1).sum()) / retained})
display(pd.DataFrame(sequence_audit).round(4))

El vocabulario de embeddings retiene como máximo 20.000 entradas, incluidos padding y desconocida, y exige dos apariciones en ajuste. Ese límite reduce parámetros frente a las 50.000 palabras de la guiada. Los primeros 512 tokens se conservan y se añade padding al final; la tabla muestra el coste real del truncado y la cobertura. La representación densa conserva las 30.000 características TF-IDF de la referencia clásica, incluidos bigramas; las diferencias entre ambas familias incluyen la representación.

`Flatten` conserva posiciones dentro de la ventana y multiplica la entrada de la primera densa por 512. El promedio enmascarado excluye el padding y produce un vector de 32 o 64 dimensiones; pierde el orden de las palabras, pero reduce parámetros. En `Flatten` el índice de padding participa en el aprendizaje y puede codificar longitud/posición. Estas diferencias se consideran al interpretar el resultado. La máscara y la restauración de pesos siguen la documentación de [Embedding](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding), [GlobalAveragePooling1D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling1D) y [EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping).

In [ ]:
DENSE_BASE = dict(family="densa", hidden=[64], activation="relu", lr=0.001,
                  optimizer="adam", batch=64, dim=0, pooling="ninguno")
EMB_BASE = dict(family="embedding", hidden=[32], activation="relu", lr=0.001,
                optimizer="adam", batch=64, dim=32, pooling="flatten")
configs = [
    dict(DENSE_BASE, name="D64"),
    dict(DENSE_BASE, name="D128-64", hidden=[128, 64]),
    dict(DENSE_BASE, name="D64-tanh", activation="tanh"),
    dict(DENSE_BASE, name="D64-lr3e-4", lr=0.0003),
    dict(DENSE_BASE, name="D64-SGD", optimizer="sgd", lr=0.1),
    dict(DENSE_BASE, name="D64-b128", batch=128),
    dict(EMB_BASE, name="E32-flat"),
    dict(EMB_BASE, name="E64-flat", dim=64),
    dict(EMB_BASE, name="E32-mean", pooling="mean"),
    dict(EMB_BASE, name="E64-mean", dim=64, pooling="mean"),
    dict(EMB_BASE, name="E32-mean-b32", pooling="mean", batch=32),
]
MAX_EPOCHS, PATIENCE, MIN_DELTA = 20, 3, 0.001

def create_network(cfg):
    if cfg["family"] == "densa":
        inputs = keras.Input(shape=(X_dense_fit.shape[1],), dtype="float32")
        x = inputs
    else:
        inputs = keras.Input(shape=(MAX_LEN,), dtype="int32")
        x = keras.layers.Embedding(VOCAB_SIZE, cfg["dim"],
            mask_zero=(cfg["pooling"] == "mean"))(inputs)
        x = (keras.layers.GlobalAveragePooling1D()(x) if cfg["pooling"] == "mean"
             else keras.layers.Flatten()(x))
    for units in cfg["hidden"]:
        x = keras.layers.Dense(units, activation=cfg["activation"],
            kernel_regularizer=keras.regularizers.l2(0.0001))(x)
        x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    opt = (keras.optimizers.Adam(learning_rate=cfg["lr"]) if cfg["optimizer"] == "adam"
           else keras.optimizers.SGD(learning_rate=cfg["lr"], momentum=0.9))
    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    return model

def dataset(X, y, batch, training=False):
    # Densificar solo un lote evita una matriz completa de 6.400 x 30.000.
    if hasattr(X, "tocoo"):
        coo = X.tocoo()
        tensor = tf.sparse.reorder(tf.SparseTensor(
            np.column_stack([coo.row, coo.col]).astype("int64"), coo.data, coo.shape))
        ds = tf.data.Dataset.from_tensor_slices((tensor, y))
    else:
        ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch)
    if hasattr(X, "tocoo"):
        ds = ds.map(lambda a,b: (tf.sparse.to_dense(a), b), num_parallel_calls=1)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 1
    options.experimental_deterministic = True
    return ds.with_options(options).prefetch(1)

class EpochProgress(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch+1) % 5 == 0:
            print(f"  época {epoch+1}, pérdida validación={logs['val_loss']:.4f}", flush=True)

nn_rows, nn_histories, nn_predictions, nn_probabilities = [], {}, {}, {}
for cfg in configs:
    keras.backend.clear_session()
    gc.collect()
    keras.utils.set_random_seed(SEED)
    model = create_network(cfg)
    Xf, Xv = ((X_dense_fit, X_dense_valid) if cfg["family"] == "densa"
              else (X_emb_fit, X_emb_valid))
    ds_fit = dataset(Xf, y_fit, cfg["batch"], training=True)
    ds_valid = dataset(Xv, y_valid, cfg["batch"])
    early = keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=PATIENCE,
        min_delta=MIN_DELTA, restore_best_weights=True)
    start = time.perf_counter()
    print("Entrenando", cfg["name"], flush=True)
    history = model.fit(ds_fit, validation_data=ds_valid, epochs=MAX_EPOCHS,
                        callbacks=[early, EpochProgress()], shuffle=False, verbose=0)
    probability = model.predict(ds_valid.map(lambda a,b: a), verbose=0).ravel()
    prediction = np.where(probability >= 0.5, "positive", "negative")
    assert np.isfinite(probability).all() and len(prediction) == len(valid)
    nn_histories[cfg["name"]] = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    nn_predictions[cfg["name"]] = prediction
    nn_probabilities[cfg["name"]] = probability
    nn_rows.append({"modelo": cfg["name"], "familia": cfg["family"],
        "capas ocultas": str(cfg["hidden"]), "activación": cfg["activation"],
        "optimizador": cfg["optimizer"], "tasa": cfg["lr"], "batch": cfg["batch"],
        "dim embedding": cfg["dim"], "agregación": cfg["pooling"],
        "parámetros": int(model.count_params()), "épocas ejecutadas": len(history.epoch),
        "época restaurada": int(early.best_epoch)+1,
        "pérdida ajuste restaurada": float(model.evaluate(dataset(Xf, y_fit, cfg["batch"]), verbose=0)[0]),
        "pérdida validación restaurada": float(model.evaluate(ds_valid, verbose=0)[0]),
        "F1 macro validación": float(f1_score(valid.sentiment, prediction, average="macro")),
        "accuracy validación": float(accuracy_score(valid.sentiment, prediction)),
        "entrenamiento y evaluación s": time.perf_counter()-start})
    print(cfg["name"], "F1", round(nn_rows[-1]["F1 macro validación"],4), flush=True)
    del model, ds_fit, ds_valid
nn_comparison = pd.DataFrame(nn_rows).sort_values(
    ["F1 macro validación", "accuracy validación"], ascending=False, kind="stable").reset_index(drop=True)
display(nn_comparison.round(4))

### Validación y comparación de variantes

Las pérdidas incluyen la penalización L2, además de la entropía cruzada. Para estudiar sobreajuste se muestran pérdidas de ajuste y validación en las épocas primera, restaurada y última de cada variante. La pérdida de ajuste registrada durante una época utiliza dropout y pesos que cambian entre lotes; las pérdidas restauradas de la tabla anterior se calculan en modo de evaluación. Los tiempos corresponden a esta ejecución y no son una comparación de eficiencia bajo condiciones idénticas entre bibliotecas.

In [ ]:
trajectory_rows = []
for row in nn_rows:
    name = row["modelo"]
    hist = nn_histories[name]
    for ep in sorted(set([0, row["época restaurada"]-1, row["épocas ejecutadas"]-1])):
        trajectory_rows.append({"modelo": name, "época": ep+1,
            "pérdida ajuste": hist["loss"][ep], "pérdida validación": hist["val_loss"][ep],
            "accuracy ajuste": hist["accuracy"][ep], "accuracy validación": hist["val_accuracy"][ep]})
display(pd.DataFrame(trajectory_rows).round(4))

classical_summary = comparison.rename(columns={"algoritmo": "modelo"}).copy()
classical_summary["modelo"] = classical_summary["modelo"] + " / " + classical_summary["representación"]
classical_summary["familia"] = "clásico"
combined = pd.concat([classical_summary, nn_comparison], ignore_index=True).sort_values(
    ["F1 macro validación", "accuracy validación"], ascending=False, kind="stable").reset_index(drop=True)
display(combined[["modelo", "familia", "F1 macro validación", "accuracy validación"]].round(4))
best_nn_name = nn_comparison.iloc[0]["modelo"]
pred_nn = nn_predictions[best_nn_name]
display(pd.DataFrame(confusion_matrix(valid.sentiment, pred_nn, labels=LABELS),
    index=["real negativa", "real positiva"], columns=["predicha negativa", "predicha positiva"]))
display(pd.DataFrame(classification_report(valid.sentiment, pred_nn, labels=LABELS, output_dict=True)).T.round(4))

D128-64 obtiene el mayor F1 macro neuronal de validación, 0,8812, y accuracy 0,88125. La referencia D64 alcanza F1 0,8781; cambiar a tanh da 0,8800 y reducir la tasa a 0,0003 da 0,8794. SGD con tasa 0,1 queda en 0,8735 y el batch 128 en 0,8725. La arquitectura más ancha y profunda mejora cinco aciertos respecto a D64, pero duplica aproximadamente los parámetros. En D128-64 se restaura la primera época y se detiene tras la cuarta: el entrenamiento posterior no mejora suficientemente la pérdida de validación. La separación entre pérdidas de ajuste y validación muestra que reducir el error de ajuste no garantiza mejorar la generalización. Las variantes de tasa reducida y batch 128 llegan al límite de 20 épocas; esta búsqueda no demuestra que hayan convergido.

En embeddings, E32-mean mejora el F1 de E32-flat de 0,8287 a 0,8669, con 61 aciertos adicionales y menos parámetros, 641.089 frente a 1.164.353. El cambio incluye la agregación y el tratamiento del padding. Pasar de 32 a 64 dimensiones no mejora ninguna de las dos agregaciones en esta ejecución. El batch 32 del promedio alcanza F1 0,8630, inferior al batch 64. La capacidad adicional del aplanado y del embedding de 64 dimensiones no se traduce en mejor validación; por ejemplo, E64-flat reduce más la pérdida de ajuste que E32-flat y obtiene peor resultado de validación.

La mejor red supera provisionalmente al mejor clásico, regresión logística con TF-IDF 1–2g, que obtiene F1 0,8750 y accuracy 0,875. La diferencia son diez aciertos entre 1.600 reseñas, del 87,50 % al 88,125 %. No se interpreta como una superioridad general: las redes usan una sola semilla y la validación se ha consultado para la parada y para comparar once variantes, además de las combinaciones clásicas. Las búsquedas de ambas familias también tienen presupuestos distintos. El TF-IDF clásico y denso usa el texto completo; los embeddings retienen solo los primeros 512 tokens, con 100 reseñas truncadas en validación. Por tanto, la comparación entre representaciones incluye diferencias de cobertura y arquitectura.

## 6. Análisis comparado de errores

Se comparan la referencia clásica y la mejor variante neuronal en las mismas 1.600 reseñas. Las cuatro combinaciones de acierto/error permiten medir qué errores comparten y cuáles corrige cada familia. Para leer casos concretos se muestran las dos primeras discrepancias de cada tipo por orden del CSV, y las dos primeras equivocaciones compartidas. La regla se fija sin elegir fragmentos por su contenido. Las probabilidades neuronales no se comparan con confianza clásica ni se consideran calibradas.

In [ ]:
paired = valid[["id", "review", "sentiment"]].copy()
paired["clásico"] = pred_valid
paired["neuronal"] = pred_nn
paired["prob positiva neuronal"] = nn_probabilities[best_nn_name]
paired["acierta clásico"] = paired["clásico"] == paired.sentiment
paired["acierta neuronal"] = paired["neuronal"] == paired.sentiment
paired["caso"] = np.select([
    paired["acierta clásico"] & paired["acierta neuronal"],
    paired["acierta clásico"] & ~paired["acierta neuronal"],
    ~paired["acierta clásico"] & paired["acierta neuronal"]],
    ["ambos aciertan", "solo clásico acierta", "solo neuronal acierta"], default="ambos fallan")
display(paired.groupby("caso").size().rename("reseñas").to_frame())
error_examples = paired[paired.caso != "ambos aciertan"].groupby("caso",sort=True).head(2).copy()
with pd.option_context("display.max_colwidth", None):
    display(error_examples[["id", "sentiment", "clásico", "neuronal", "caso", "review"]])
# Relación descriptiva con la longitud, sin cambiar las reglas de selección.
paired["tokens"] = [len(r) for r in valid_tokens]
paired["longitud"] = np.where(paired.tokens > MAX_LEN, "más de 512 tokens", "hasta 512 tokens")
length_errors = paired.groupby("longitud").agg(reseñas=("id", "size"),
    acierto_clásico=("acierta clásico", "mean"), acierto_neuronal=("acierta neuronal", "mean"))
display(length_errors.round(4))
print("Test reservado: no se han obtenido predicciones ni métricas sobre sus 2.000 reseñas.")

Los dos modelos aciertan en 1.367 reseñas y fallan en 157. La red corrige 43 errores del clásico e introduce 33 errores que el clásico evitaba; el saldo son diez aciertos más. Su matriz de confusión contiene 102 negativas clasificadas como positivas y 88 positivas clasificadas como negativas. La mejora agregada convive con errores nuevos y no resuelve los 157 casos compartidos.

Entre las discrepancias, la reseña 500 cita elogios ajenos antes de rechazarlos y calificar la película como poco interesante; la red acierta la etiqueta negativa. En la 5812, también negativa y corregida por la red, una descripción extensa de la trama termina en críticas al ritmo y al desenlace. En cambio, la 10701 comienza recordando otra película disfrutada y después critica la actual; acierta el clásico y falla la red. La 21249 tiene etiqueta positiva, pero expresa indignación política y sufrimiento al comentar el tema de la película. Ilustra la diferencia entre tono emocional y valoración de la película; la red la clasifica como negativa. Estas lecturas describen el contenido de los errores, sin demostrar qué expresiones causaron la predicción.

Los errores compartidos 2895 y 19118 ya aparecían en la referencia clásica. El primero mezcla recuerdos del rodaje con críticas, y el segundo usa elogios entre comillas junto a juicios negativos. La red densa sigue usando una representación léxica sin una estructura explícita de discurso; añadir capas no basta para resolver esos ejemplos. Las tasas por longitud son descriptivas: el grupo de más de 512 tokens tiene solo 100 reseñas y la red analizada es densa, por lo que esa tabla no mide directamente el efecto causal del truncado en embeddings.

## 7. Selección y evaluación final

Se fija D128-64 con TF-IDF de unigramas y bigramas, la configuración con mayor F1 macro de la comparación de validación. Se mantienen dos capas ocultas de 128 y 64 unidades ReLU, L2 de 0,0001, dropout de 0,3, salida sigmoide, entropía cruzada binaria, Adam con tasa 0,001, batch 64 y umbral 0,5. La selección incluye la limpieza principal y el vocabulario máximo de 30.000 características con frecuencia documental mínima de dos.

Se reajustan vocabulario, IDF y pesos con las 8.000 reseñas de entrenamiento y validación. El número de épocas queda fijado en una, la época restaurada de esta red durante la validación. No se reservan de nuevo datos para parada, ni se consultan las etiquetas de test durante el ajuste. Reaprender la representación y ampliar los datos cambia el modelo: una época es una regla de traslado fijada con validación, no una garantía de convergencia u optimalidad.

A continuación se realiza la única evaluación final sobre las 2.000 reseñas de test. No se comparan otros modelos con ese resultado ni se modifican hiperparámetros después de observarlo. Los ocho textos compartidos entre los archivos docentes limitan su independencia; se conserva la separación oficial y no se presenta la puntuación como una medida sobre textos totalmente inéditos.

In [ ]:
# Decisiones cerradas con validación, antes de calcular predicciones de test.
FINAL_NAME, FINAL_EPOCHS, FINAL_THRESHOLD = "D128-64", 1, 0.5
assert combined.iloc[0]["modelo"] == FINAL_NAME
selected_cfg = next(dict(c) for c in configs if c["name"] == FINAL_NAME)
selected_validation = next(r for r in nn_rows if r["modelo"] == FINAL_NAME)
assert selected_validation["época restaurada"] == FINAL_EPOCHS
final_vectorizer = clone(representaciones["TF-IDF 1-2g"])
X_final_train = final_vectorizer.fit_transform(train.clean).astype("float32")
X_final_test = final_vectorizer.transform(test.clean).astype("float32")
assert final_vectorizer.vocabulary_ == clone(representaciones["TF-IDF 1-2g"]).fit(train.clean).vocabulary_
# create_network toma esta dimensión; las funciones y parámetros de la red se conservan.
assert X_final_train.shape[1] == X_dense_fit.shape[1]
keras.backend.clear_session()
gc.collect()
keras.utils.set_random_seed(SEED)
final_model = create_network(selected_cfg)
y_final_train = (train.sentiment == "positive").astype("float32").to_numpy()
final_history = final_model.fit(dataset(X_final_train, y_final_train, selected_cfg["batch"], training=True),
    epochs=FINAL_EPOCHS, shuffle=False, verbose=0)
# El predictor solo recibe características; las etiquetas se usan después para medir.
final_probability = final_model.predict(dataset(X_final_test, np.zeros(len(test), dtype="float32"),
    selected_cfg["batch"]).map(lambda a,b: a), verbose=0).ravel()
final_prediction = np.where(final_probability >= FINAL_THRESHOLD, "positive", "negative")
assert len(final_prediction) == 2000 and np.isfinite(final_probability).all()
final_metrics = {"modelo": FINAL_NAME, "reseñas entrenamiento": len(train),
    "reseñas test": len(test), "épocas reajuste": FINAL_EPOCHS,
    "F1 macro test": float(f1_score(test.sentiment, final_prediction, average="macro")),
    "accuracy test": float(accuracy_score(test.sentiment, final_prediction))}
final_confusion = confusion_matrix(test.sentiment, final_prediction, labels=LABELS)
final_report = classification_report(test.sentiment, final_prediction, labels=LABELS, output_dict=True)
display(pd.DataFrame([final_metrics]).round(4))
display(pd.DataFrame(final_confusion, index=["real negativa", "real positiva"],
    columns=["predicha negativa", "predicha positiva"]))
display(pd.DataFrame(final_report).T.round(4))


El modelo final acierta 1.749 de las 2.000 reseñas de test, con accuracy de 0,8745 y F1 macro de 0,8742. Clasifica como negativas 149 reseñas positivas y como positivas 102 negativas. El recall es menor en la clase positiva, 0,8472 frente a 0,9005 en la negativa, aunque su precisión es mayor, 0,8901 frente a 0,8610. Por tanto, los errores no se reparten por igual entre las dos clases.

El F1 macro de test queda por debajo del 0,8812 de validación. Las particiones son distintas y el modelo final reaprende tanto la representación como los pesos con más datos; esta diferencia no permite aislar el efecto de cada cambio. La puntuación de validación también había intervenido en la búsqueda y en la parada. El resultado final aporta una evaluación separada de esas decisiones, con la limitación de los textos repetidos ya descrita.